# YOLO11s → cvimodel Conversion for MaixCAM (CV181x)

**เป้าหมาย:** Export YOLO11s pre-trained model เป็น cvimodel (INT8) สำหรับ MaixCAM  
**Method:** B (2 output nodes) — รองรับโดย `nn.YOLO11` บน MaixCAM  
**Input size:** 224×320  
**Processor:** cv181x

In [ ]:
# Step 1: Install condacolab
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...


In [ ]:
# Step 2: Clone server repo + setup conda env + install tpu-mlir
!git clone https://github.com/KidBrightAI/kidbright_MAI_server server
%cd server

# extract tools
!tar -xf tools/opencv-3.4.13.tar.gz -C /
!chmod +x tools/onnx2ncnn
!chmod +x tools/spnntools
!conda create -n kbmai python=3.10 -y
!wget https://github.com/sophgo/tpu-mlir/releases/download/v1.27/tpu_mlir-1.27-py3-none-any.whl

Cloning into 'server'...
remote: Enumerating objects: 1622, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 1622 (delta 57), reused 63 (delta 25), pack-reused 1519 (from 1)
Receiving objects: 100% (1622/1622), 234.88 MiB | 31.30 MiB/s, done.
Resolving deltas: 100% (697/697), done.
Updating files: 100% (1184/1184), done.
/content/server
Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.3.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/kbmai

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  c

In [ ]:
%%bash
source activate kbmai
pip install -q tpu_mlir*.whl

Reason for being yanked: deprecated, use 4.8.0.76


In [ ]:
# Step 3: Install Python dependencies
!conda run -n kbmai pip -q install "setuptools<70.0.0" psutil flatbuffers "onnx==1.14.1" "onnxruntime==1.16.3" "onnxruntime_extensions==0.14.0" "torch==2.1.0" protobuf flask torchvision torchsummary
!conda run -n kbmai pip install "onnxsim==0.4.17" "onnxslim>=0.1.71" onnxruntime-gpu
!conda run -n kbmai pip install ultralytics "numpy<2" matplotlib-inline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 6.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 64.5 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 138.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.9 MB/s  0:00:00


INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 55.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 kB 36.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 74.7 MB/s  0:00:00




## YOLO11s Export + Conversion

In [ ]:
# Step 4: Download yolo11s.pt and export to ONNX
!wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11s.pt
!conda run -n kbmai --live-stream python -c "from ultralytics import YOLO; model = YOLO('yolo11s.pt'); model.export(format='onnx', imgsz=[224, 320], opset=16); print('Export done!')"

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.36 🚀 Python-3.10.20 torch-2.1.0+cu121 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s summary (fused): 100 layers, 9,443,760 parameters, 0 gradients, 21.5 GFLOPs

PyTorch: starting from 'yolo11s.pt' with input shape (1, 3, 224, 320) BCHW and output shape(s) (1, 84, 1470) (18.4 MB)

ONNX: starting export with onnx 1.14.1 opset 16...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 2.4s, saved as 'yolo11s.onnx' (36.2 MB)

Export complete (3.0s)
Results saved to /content/server
Predict:         yolo predict task=detect model

In [ ]:
# Step 5: Inspect ONNX output node names
!conda run -n kbmai --live-stream python -c "import onnx; model = onnx.load('yolo11s.onnx'); print('=== Output Nodes ==='); [print(f'  {o.name}: {[d.dim_value for d in o.type.tensor_type.shape.dim]}') for o in model.graph.output]; print(); print('=== Nodes containing model.23 ==='); [print(f'  {n.op_type}: {o}') for n in model.graph.node if 'model.23' in n.name for o in n.output]"

=== Output Nodes ===
  output0: [1, 84, 1470]

=== Nodes containing model.23 ===
  Conv: /model.23/cv2.0/cv2.0.0/conv/Conv_output_0
  Conv: /model.23/cv3.0/cv3.0.0/cv3.0.0.0/conv/Conv_output_0
  Sigmoid: /model.23/cv2.0/cv2.0.0/act/Sigmoid_output_0
  Sigmoid: /model.23/cv3.0/cv3.0.0/cv3.0.0.0/act/Sigmoid_output_0
  Mul: /model.23/cv2.0/cv2.0.0/act/Mul_output_0
  Mul: /model.23/cv3.0/cv3.0.0/cv3.0.0.0/act/Mul_output_0
  Conv: /model.23/cv2.0/cv2.0.1/conv/Conv_output_0
  Conv: /model.23/cv3.0/cv3.0.0/cv3.0.0.1/conv/Conv_output_0
  Sigmoid: /model.23/cv2.0/cv2.0.1/act/Sigmoid_output_0
  Sigmoid: /model.23/cv3.0/cv3.0.0/cv3.0.0.1/act/Sigmoid_output_0
  Mul: /model.23/cv2.0/cv2.0.1/act/Mul_output_0
  Mul: /model.23/cv3.0/cv3.0.0/cv3.0.0.1/act/Mul_output_0
  Conv: /model.23/cv2.0/cv2.0.2/Conv_output_0
  Conv: /model.23/cv3.0/cv3.0.1/cv3.0.1.0/conv/Conv_output_0
  Reshape: /model.23/Reshape_output_0
  Sigmoid: /model.23/cv3.0/cv3.0.1/cv3.0.1.0/act/Sigmoid_output_0
  Mul: /model.23/cv3.0/cv3.0

## Method B Conversion (2 output nodes)

**สำคัญ:** ตรวจสอบ output node names จาก Step 5 ก่อนรัน  
- ถ้า yolo11s ใช้ `/model.23/...` เหมือน yolo11n → รันได้เลย  
- ถ้าต่าง → แก้ `--output_names` ใน cell ด้านล่างให้ตรง

In [ ]:
# Step 6: model_transform — Method B (2 output nodes)
# output_names อาจต้องแก้ตาม Step 5
!conda run -n kbmai model_transform.py \
  --model_name yolo11s \
  --model_def /content/server/yolo11s.onnx \
  --input_shapes [[1,3,224,320]] \
  --mean "0,0,0" \
  --output_names "/model.23/dfl/conv/Conv_output_0,/model.23/Sigmoid_output_0" \
  --scale "0.00392156862745098,0.00392156862745098,0.00392156862745098" \
  --keep_aspect_ratio \
  --pixel_format rgb \
  --channel_format nchw \
  --test_input ./data/test_images2/cat.jpg \
  --test_result yolo11s_top_outputs.npz \
  --tolerance 0.99,0.99 \
  --mlir yolo11s.mlir

2026/04/09 08:33:14 - INFO : TPU-MLIR v1.27-20260206
2026/04/09 08:33:14 - INFO : 
	 _____________________________________________________ 
	| preprocess:                                           |
	|   (x - mean) * scale                                  |
	'-------------------------------------------------------'
  config Preprocess args : 
	resize_dims           : same to net input dims
	keep_aspect_ratio     : True
	keep_ratio_mode       : letterbox
	pad_value             : 0
	pad_type              : center
	--------------------------
	mean                  : [0.0, 0.0, 0.0]
	scale                 : [0.003921569, 0.003921569, 0.003921569]
	--------------------------
	pixel_format          : rgb
	channel_format        : nchw
	yuv_type              : 

2026/04/09 08:33:20 - INFO : Input_shape assigned
2026/04/09 08:33:20 - INFO : ConstantFolding finished
2026/04/09 08:33:20 - INFO : skip_fuse_bn:False
2026/04/09 08:33:21 - INFO : Onnxsim opt finished
2026/04/09 08:33:21 - INFO : Cons

In [ ]:
# Step 7: Calibration
!conda run -n kbmai run_calibration.py yolo11s.mlir \
  --dataset /content/server/data/test_images \
  --input_num 24 \
  -o yolo11s_cali_table

TPU-MLIR v1.27-20260206
2026/04/09 08:34:20 - INFO : 
  load_config Preprocess args : 
	resize_dims           : [224, 320]
	keep_aspect_ratio     : True
	keep_ratio_mode       : letterbox
	pad_value             : 0
	pad_type              : center
	input_dims            : [224, 320]
	--------------------------
	mean                  : [0.0, 0.0, 0.0]
	scale                 : [0.003921569, 0.003921569, 0.003921569]
	--------------------------
	pixel_format          : rgb
	channel_format        : nchw
	yuv_type              : 

input_num = 24, ref = 24
real input_num = 24
auto tune end, run time:183.7071750164032


activation_collect_and_calc_th for sample: 0:   0%|          | 0/24 [00:00<?, ?it/s][                                                  ] 0%                                                  ] 0%                                                  ] 0%

In [ ]:
# Step 8: Deploy to cvimodel (INT8, cv181x)
# ลด tolerance เพราะ yolo11s quantize แล้ว Sigmoid node ค่า euclidean ต่ำกว่า yolo11n
!conda run -n kbmai model_deploy.py \
  --mlir yolo11s.mlir \
  --quantize INT8 \
  --calibration_table yolo11s_cali_table \
  --processor cv181x \
  --test_input yolo11s_in_f32.npz \
  --test_reference yolo11s_top_outputs.npz \
  --tolerance 0.85,0.5 \
  --model yolo11s.cvimodel

2026/04/09 08:38:42 - INFO : TPU-MLIR v1.27-20260206


[/model.23/dfl/conv/Conv_output_0_Conv]      SIMILAR [PASSED]
    (1, 1, 4, 1470) float32 
    cosine_similarity      = 0.996624
    euclidean_similarity   = 0.916316
    sqnr_similarity        = 15.918308
[/model.23/Sigmoid_output_0_Sigmoid]      SIMILAR [PASSED]
    (1, 80, 1470) float32 
    cosine_similarity      = 0.905014
    euclidean_similarity   = 0.553287
    sqnr_similarity        = 7.165079
2 compared
2 passed
  0 equal, 0 close, 2 similar
0 failed
  0 not equal, 0 not similar
min_similiarity = (0.9050138592720032, 0.5532872917354736, 7.165079116821289)
Target    yolo11s_cv181x_int8_sym_tpu_outputs.npz
Reference yolo11s_top_outputs.npz
npz compare PASSED.
Dumping LgConfig!
Global Configs:
  shape_secs_search_strategy = 0
  structure_detect_opt = 1
Search Method Config: sc_method_quick_search
  CSECS_SEARCH_RECORD_THRESHOLD = -1
  DSECS_SEARCH_RECORD_THRESHOLD = -1
  HSECS_SEARCH_RECORD_THRESHOLD = -1
  MAX_CSECS = 32
  

In [ ]:
# Step 9: Check output file size
!ls -lh yolo11s.cvimodel
!echo "---"
!ls -lh yolo11s.onnx

-rw-r--r-- 1 root root 9.6M Apr  9 08:38 yolo11s.cvimodel
---
-rw-r--r-- 1 root root 37M Apr  9 08:32 yolo11s.onnx


In [ ]:
# Step 10b: Copy to Google Drive for easy access
from google.colab import drive
drive.mount('/content/drive')
!cp yolo11s.cvimodel /content/drive/MyDrive/
!echo "Copied yolo11s.cvimodel to Google Drive"

In [ ]:
# Step 10: Download cvimodel
from google.colab import files
files.download('yolo11s.cvimodel')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>